# 01 - XRF HDF5 데이터 로딩

이 노트북은 `h5py`를 사용하여 MAPS HDF5 형식으로 저장된 X선 형광(XRF) 데이터를
로드하는 방법을 시연합니다. 다루는 내용:

1. HDF5 파일 열기 및 그룹 계층 구조 검사
2. 사용 가능한 원소 목록 확인
3. 원소별 기본 통계 계산
4. 원소 맵 시각화

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

## 1. MAPS HDF5 파일 열기

MAPS는 피팅된 원소 맵을 `/MAPS/XRF_fits` 아래에 저장합니다. 해당 그룹의 각 데이터셋은
2D 원소 농도 맵에 해당합니다.

In [ ]:
DATA_PATH = "../data/sample_xrf.h5"  # adjust to your file

f = h5py.File(DATA_PATH, "r")

def print_tree(group, indent=0):
    """Recursively print HDF5 group hierarchy."""
    for key in group:
        item = group[key]
        prefix = "  " * indent
        if isinstance(item, h5py.Group):
            print(f"{prefix}[Group] {key}")
            print_tree(item, indent + 1)
        else:
            print(f"{prefix}[Dataset] {key}  shape={item.shape}  dtype={item.dtype}")

print_tree(f)

## 2. 사용 가능한 원소 목록

`channel_names` 데이터셋은 `XRF_fits`의 첫 번째 축 슬라이스에 대응하는
원소 기호를 저장합니다.

In [ ]:
xrf_group = f["/MAPS"]
channel_names = [name.decode() for name in xrf_group["channel_names"][:]]
xrf_fits = xrf_group["XRF_fits"]  # shape: (n_elements, rows, cols)

print(f"Number of elements: {len(channel_names)}")
print(f"Elements: {channel_names}")
print(f"Map shape per element: {xrf_fits.shape[1:]}")

## 3. 원소별 기본 통계

In [ ]:
data = xrf_fits[:]  # load into memory

print(f"{'Element':<8} {'Mean':>12} {'Std':>12} {'Min':>12} {'Max':>12}")
print("-" * 60)
for i, name in enumerate(channel_names):
    arr = data[i]
    print(f"{name:<8} {arr.mean():12.4f} {arr.std():12.4f} {arr.min():12.4f} {arr.max():12.4f}")

## 4. 원소 맵 시각화

원소 농도 맵 그리드를 표시하여 공간 분포에 대한 초기 감각을 얻습니다.

In [ ]:
n_elements = len(channel_names)
ncols = 4
nrows = int(np.ceil(n_elements / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.5 * nrows))
axes = axes.flatten()

for i, name in enumerate(channel_names):
    im = axes[i].imshow(data[i], cmap="inferno")
    axes[i].set_title(name)
    axes[i].axis("off")
    fig.colorbar(im, ax=axes[i], fraction=0.046, pad=0.04)

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
f.close()